<a href="https://colab.research.google.com/github/fvangool/Deep-Learning-Specialization-Coursera/blob/main/GNN_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Run these first in Colab
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install cuml-cu12  # This is the big one for KNN

Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached https://data.pyg.org/whl/torch-2.4.0%2Bcu121/torch_scatter-2.1.2%2Bpt24cu121-cp312-cp312-linux_x86_64.whl (10.9 MB)
  Using cached https://data.pyg.org/whl/torch-2.4.0%2Bcu121/torch_sparse-0.6.18%2Bpt24cu121-cp312-cp312-linux_x86_64.whl (5.1 MB)
  Using cached https://data.pyg.org/whl/torch-2.4.0%2Bcu121/torch_cluster-1.6.3%2Bpt24cu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 68.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82


In [3]:
# -*- coding: utf-8 -*-
"""GNN v2 Enhanced — Irrigation Need
Base improvements (always active):
  1. Decision Stump Thresholding    — binary flag per numeric feature
  2. Rank Percentile Encoding       — replaces StandardScaler
  3. Digit-Level Features           — decimal digit as categorical
  4. Autoencoder Bottleneck         — 4-dim latent numeric summary
  5. BalancedAccuracyTracker        — early stopping on val_ba

Feature-flagged additions (toggle in FEATURE FLAGS block):
  F1. USE_TTA          — Test-Time Augmentation (multi-offset inference avg)
  F2. USE_LABEL_SMOOTH — Label Smoothing on CrossEntropyLoss
  F3. USE_MIXUP        — Mixup augmentation in embedding space
  F4. USE_COSINE_LR    — Cosine Annealing LR schedule
  F5. USE_SSL_PRETRAIN — Self-Supervised masked-feature pre-training
  F6. SAVE_STACKER_OOF — Save raw (untuned) OOF separately for the meta-learner
"""

# ---- Colab setup ----
from google.colab import drive
drive.mount('/content/drive')

!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install cuml-cu12

import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.tree import DecisionTreeClassifier

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

try:
    from cuml.neighbors import NearestNeighbors
    print("Using RAPIDS cuML KNN - Very Fast!")
except ImportError:
    from sklearn.neighbors import NearestNeighbors
    print("cuML not found, falling back to sklearn KNN (slower)")


# ============================================================
# FEATURE FLAGS  <- toggle experiments here, one at a time
# ============================================================
USE_TTA          = False   # F1: disabled — high FANOUT/K ratio limits diversity gain
                           #     re-enable with fanouts=[4,3] for more sampling variance
USE_LABEL_SMOOTH = True   # F2: label smoothing on CrossEntropyLoss (try 0.05 first)
USE_MIXUP        = True   # F3: mixup in embedding space after lin_in (try alpha=0.4)
USE_COSINE_LR    = True   # F4: cosine annealing LR (replaces flat LR)
USE_SSL_PRETRAIN = True   # F5: self-supervised masked pre-training before CV loop
SAVE_STACKER_OOF = True    # F6: save raw OOF as oof_stacker_<VERSION>.npy


# ============================================================
# CONFIG
# ============================================================
OUT_DIR     = "/content/drive/MyDrive/irrigation_need_v15/"
TRAIN_PATH  = "/content/drive/MyDrive/irrigation_need_v15/train.csv"
TEST_PATH   = "/content/drive/MyDrive/irrigation_need_v15/test.csv"
VERSION_NB  = "GNN_v4_enhanced"
SEED        = 42
N_FOLDS     = 5
K           = 8
EPOCHS      = 80
PATIENCE    = 15
BATCH_SIZE  = 4096
INFER_BATCH = 8192
FANOUTS     = [6, 4]
GRAPH_NUM_MULTIPLIER = 3.0
USE_AMP     = True
RARE_MIN    = 25
HIDDEN      = 128
DROPOUT     = 0.20
LR          = 8e-4
WEIGHT_DECAY = 3e-4

# F1 — TTA
TTA_PASSES = 8              # stochastic inference passes to average

# F2 — Label Smoothing
LABEL_SMOOTHING = 0.05      # range 0.03–0.10

# F3 — Mixup
MIXUP_ALPHA = 0.4           # Beta(alpha, alpha); range 0.2–0.6

# F4 — Cosine LR
COSINE_ETA_MIN = 1e-5       # minimum LR at end of annealing

# F5 — SSL pre-training
SSL_EPOCHS     = 20         # keep light; 15–30 is typical
SSL_MASK_RATIO = 0.25       # fraction of numeric features masked per node
SSL_LR         = 5e-4

# Autoencoder (base, always active)
AE_LATENT_DIM = 4
AE_EPOCHS     = 30
AE_BATCH_SIZE = 2048
AE_LR         = 1e-3

TARGET    = "Irrigation_Need"
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2}
LABEL_INV = {0: "Low", 1: "Medium", 2: "High"}
N_CLASSES = 3

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm"
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region"
]

CAT_PROXY  = [f"{c}__cat"     for c in NUMS]
CAT_RARE   = [f"{c}__is_rare" for c in NUMS]
STUMP_COLS = [f"{c}__stump"   for c in NUMS]
DIGIT_COLS = [f"{c}__digit"   for c in NUMS]
AE_COLS    = [f"ae_feat_{i}"  for i in range(AE_LATENT_DIM)]

ALL_CATS = CATS + CAT_PROXY + CAT_RARE + STUMP_COLS + DIGIT_COLS
ALL_NUMS = NUMS[:] + AE_COLS

GRAPH_CAT_COLS = CATS[:]
GRAPH_NUM_COLS = NUMS[:]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")

N_GPUS = torch.cuda.device_count()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Print active flags at startup
print("\n" + "="*60)
print("  ACTIVE FEATURE FLAGS")
print("="*60)
for _name, _val in [
    ("USE_TTA",          USE_TTA),
    ("USE_LABEL_SMOOTH", USE_LABEL_SMOOTH),
    ("USE_MIXUP",        USE_MIXUP),
    ("USE_COSINE_LR",    USE_COSINE_LR),
    ("USE_SSL_PRETRAIN", USE_SSL_PRETRAIN),
    ("SAVE_STACKER_OOF", SAVE_STACKER_OOF),
]:
    print(f"  {'ON ' if _val else 'OFF'}  {_name}")
print("="*60 + "\n")


# ============================================================
# BASE IMPROVEMENT 1 — DECISION STUMP THRESHOLDING
# ============================================================

def build_decision_stumps(train_df: pd.DataFrame, y: np.ndarray) -> dict:
    """
    Fit DecisionTreeClassifier(max_depth=1) per numeric column.
    Returns col -> threshold. Fit on train only; applied to train+test.
    Hard binary signals that linear meta-stackers cannot derive from
    continuous features alone.
    """
    stumps = {}
    for col in NUMS:
        x  = train_df[col].values.reshape(-1, 1).astype(np.float32)
        dt = DecisionTreeClassifier(max_depth=1, random_state=SEED)
        dt.fit(x, y)
        stumps[col] = float(dt.tree_.threshold[0])
    return stumps


def apply_decision_stumps(df: pd.DataFrame, stumps: dict) -> pd.DataFrame:
    df = df.copy()
    for col, thr in stumps.items():
        df[f"{col}__stump"] = (df[col].astype(np.float32) >= thr).astype(np.int8)
    return df


# ============================================================
# BASE IMPROVEMENT 2 — RANK PERCENTILE ENCODING
# ============================================================

class RankPercentileEncoder:
    """
    Converts each numeric column to rank percentile in [0, 1].
    Immune to outliers; stretches dense clusters.
    Equivalent to GaussRank without the Gaussian transform.
    """
    def __init__(self):
        self.sorted_values_ = {}

    def fit(self, df: pd.DataFrame, cols: list):
        for col in cols:
            vals = df[col].values.astype(np.float32)
            self.sorted_values_[col] = np.sort(vals[~np.isnan(vals)])
        return self

    def transform(self, df: pd.DataFrame, cols: list) -> np.ndarray:
        out = np.zeros((len(df), len(cols)), dtype=np.float32)
        for j, col in enumerate(cols):
            vals  = df[col].values.astype(np.float32)
            sv    = self.sorted_values_[col]
            ranks = np.searchsorted(sv, vals, side="left").astype(np.float32)
            out[:, j] = ranks / max(len(sv), 1)
        return out

    def fit_transform(self, df: pd.DataFrame, cols: list) -> np.ndarray:
        self.fit(df, cols)
        return self.transform(df, cols)


# ============================================================
# BASE IMPROVEMENT 3 — DIGIT-LEVEL FEATURES
# ============================================================

def add_digit_features(df: pd.DataFrame, cols: list, k: int = 1) -> pd.DataFrame:
    """
    Extracts k-th decimal digit: floor(val * 10^k + 1e-9) % 10.
    Decimal precision often reflects sensor quantisation states
    that correlate with the target.  k=1 gives first decimal digit.
    """
    df = df.copy()
    for col in cols:
        vals  = pd.to_numeric(df[col], errors="coerce").astype(np.float64)
        digit = (np.floor(vals * (10 ** k) + 1e-9) % 10).astype(np.int8)
        df[f"{col}__digit"] = digit.astype(str)
    return df


# ============================================================
# BASE IMPROVEMENT 4 — AUTOENCODER BOTTLENECK FEATURES
# ============================================================

class _Autoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),         nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train_autoencoder(
    Xn_train:   np.ndarray,
    Xn_test:    np.ndarray,
    latent_dim: int   = AE_LATENT_DIM,
    epochs:     int   = AE_EPOCHS,
    batch_size: int   = AE_BATCH_SIZE,
    lr:         float = AE_LR,
    device:     torch.device = DEVICE,
) -> tuple:
    """
    Unsupervised AE trained on combined train+test numeric features.
    Bottleneck gives the stacker a compressed environment summary that
    no individual feature captures on its own.
    """
    print(f"\n[AE] Training  latent={latent_dim}  epochs={epochs}")
    Xn_all   = np.vstack([Xn_train, Xn_test]).astype(np.float32)
    X_tensor = torch.tensor(Xn_all, dtype=torch.float32)
    model    = _Autoencoder(input_dim=Xn_all.shape[1], latent_dim=latent_dim).to(device)
    opt      = torch.optim.Adam(model.parameters(), lr=lr)
    dl       = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_tensor),
        batch_size=batch_size, shuffle=True
    )
    model.train()
    for epoch in range(1, epochs + 1):
        total = 0.0
        for (bx,) in dl:
            bx = bx.to(device)
            recon, _ = model(bx)
            loss = F.mse_loss(recon, bx)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(bx)
        if epoch % 5 == 0 or epoch == 1:
            print(f"  AE Epoch {epoch:03d} | MSE: {total / len(Xn_all):.6f}")
    model.eval()
    with torch.no_grad():
        _, z_all = model(X_tensor.to(device))
    z_all = z_all.cpu().numpy().astype(np.float32)
    del model, opt, dl, X_tensor; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    n_tr = Xn_train.shape[0]
    print(f"[AE] Done. z shape: {z_all.shape}")
    return z_all[:n_tr], z_all[n_tr:]


# ============================================================
# BASE IMPROVEMENT 5 — BALANCED ACCURACY TRACKER
# ============================================================

class BalancedAccuracyTracker:
    """
    Accumulates confusion matrix during training and computes BA.
    Used as early-stopping signal so training does not stop merely
    because the dominant Low class is well-learned.
    """
    def __init__(self, n_classes: int = N_CLASSES):
        self.n_classes = n_classes
        self.reset()

    def reset(self):
        self.confusion = np.zeros((self.n_classes, self.n_classes), dtype=np.int64)

    def update(self, probs: np.ndarray, labels: np.ndarray):
        preds = np.argmax(probs, axis=1)
        for t, p in zip(labels, preds):
            self.confusion[int(t), int(p)] += 1

    def compute(self) -> float:
        recalls = []
        for i in range(self.n_classes):
            rs = self.confusion[i].sum()
            recalls.append(self.confusion[i, i] / rs if rs > 0 else 0.0)
        return float(np.mean(recalls))


# ============================================================
# F5 — SELF-SUPERVISED PRE-TRAINING
# ============================================================

def pretrain_ssl(
    model:      nn.Module,
    x_num_cpu:  torch.Tensor,
    x_cat_cpu:  torch.Tensor,
    neighbors:  np.ndarray,
    n_all:      int,
    device:     torch.device = DEVICE,
    epochs:     int   = SSL_EPOCHS,
    mask_ratio: float = SSL_MASK_RATIO,
    lr:         float = SSL_LR,
    batch_size: int   = BATCH_SIZE,
) -> nn.Module:
    """
    F5 — Masked feature reconstruction pre-training.

    Randomly zeros out mask_ratio of numeric features per node, then
    asks the GNN encoder (lin_in + SAGE layers) to reconstruct the
    original values via a temporary MSE head.  Only masked positions
    contribute to the loss, so unmasked features act as context.

    Why it helps: with ~3.3% High examples, the supervised signal is
    sparse. SSL bootstraps the encoder into a better initialisation
    before the supervised CV loop, improving High-class embeddings most.

    The MSE head is discarded after pre-training; only encoder weights
    (lin_in, conv1, conv2, norm1, norm2) carry over to supervised training.

    Toggle: USE_SSL_PRETRAIN = True
    """
    print(f"\n[SSL] Pre-training  epochs={epochs}  mask_ratio={mask_ratio}")
    num_dim    = x_num_cpu.shape[1]
    recon_head = nn.Linear(HIDDEN, num_dim).to(device)
    params     = list(model.parameters()) + list(recon_head.parameters())
    opt        = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)

    global _global_pos
    _global_pos = np.full(n_all, -1, dtype=np.int32)
    y_dummy     = torch.full((n_all,), -1, dtype=torch.long).pin_memory()
    all_nodes   = np.arange(n_all, dtype=np.int32)

    model.train(); recon_head.train()
    for epoch in range(1, epochs + 1):
        np.random.shuffle(all_nodes)
        epoch_loss = []
        offset     = epoch % K

        for start in range(0, len(all_nodes), batch_size):
            seeds = all_nodes[start: start + batch_size]
            batch = _build_subgraph(
                seeds, neighbors, x_num_cpu, x_cat_cpu,
                y_dummy, FANOUTS, device, offset
            )

            seed_loc    = batch.seed_local
            x_num_orig  = batch.x_num[seed_loc].clone()           # [S, num_dim] target
            mask        = torch.rand_like(x_num_orig) < mask_ratio
            x_num_mod   = batch.x_num.clone()
            x_num_mod[seed_loc] = x_num_orig.masked_fill(mask, 0.0)
            batch.x_num = x_num_mod

            # Run encoder manually (bypass classification head)
            x  = torch.cat([batch.x_num, model.cat(batch.x_cat)], dim=1)
            x  = F.dropout(F.relu(model.lin_in(x)), p=DROPOUT, training=True)
            x1 = F.relu(model.norm1(model.conv1(x,  batch.edge_index)))
            x1 = F.dropout(x1, p=DROPOUT, training=True)
            x  = x + 0.5 * x1
            x2 = F.relu(model.norm2(model.conv2(x, batch.edge_index)))
            x  = x + 0.5 * x2

            recon = recon_head(x[seed_loc])                        # [S, num_dim]
            loss  = F.mse_loss(recon[mask], x_num_orig[mask])      # masked positions only

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            epoch_loss.append(loss.item())
            del batch, x, x1, x2, recon, loss

        print(f"  SSL Epoch {epoch:03d} | Recon MSE: {np.mean(epoch_loss):.6f}")

    del recon_head, opt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("[SSL] Done. Encoder weights transferred to supervised model.\n")
    return model


# ============================================================
# F3 — MIXUP HELPERS
# ============================================================

def _mixup_batch(x_num, x_cat_emb, y_long, alpha=MIXUP_ALPHA):
    """
    F3 — Mixup in embedding space.

    Interpolates pairs of (numeric input, cat embedding, label) with
    lambda ~ Beta(alpha, alpha).  Applied only to seed nodes so the
    graph topology (edge_index) stays intact.  Returns mixed tensors
    plus (lam, y_a, y_b) for computing the blended CE loss.
    """
    n   = x_num.shape[0]
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(n, device=x_num.device)
    return (
        lam * x_num     + (1 - lam) * x_num[idx],
        lam * x_cat_emb + (1 - lam) * x_cat_emb[idx],
        lam,
        y_long,
        y_long[idx],
    )


def _mixup_loss(criterion, logits, lam, y_a, y_b):
    """Blended CE loss: lam * CE(y_a) + (1-lam) * CE(y_b)."""
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


# ============================================================
# THRESHOLD TUNING HELPERS
# ============================================================

def apply_thresholds(probs, thresholds):
    return np.argmax(probs - thresholds, axis=1)

def _fitness_thresh(thresholds, probs, y):
    return -balanced_accuracy_score(y, apply_thresholds(probs, thresholds))

def optimize_thresholds(probs, y, pop_size=30, generations=40, mutation_scale=0.05):
    """Genetic search directly maximising balanced accuracy."""
    n_classes  = probs.shape[1]
    population = [np.random.uniform(-0.2, 0.2, size=n_classes) for _ in range(pop_size)]
    for gen in range(generations):
        scores = np.array([_fitness_thresh(ind, probs, y) for ind in population])
        idx    = np.argsort(scores)
        population = [population[i] for i in idx[:pop_size // 2]]
        children   = []
        while len(children) < pop_size // 2:
            p1, p2 = random.sample(population, 2)
            child  = (p1 + p2) / 2 + np.random.normal(0, mutation_scale, size=n_classes)
            children.append(child)
        population.extend(children)
        print(f"Gen {gen:02d} | Best BA: {-scores[idx[0]]:.5f}")
    scores = np.array([_fitness_thresh(ind, probs, y) for ind in population])
    return population[np.argmin(scores)]


# ============================================================
# PREPROCESSING
# ============================================================

def preprocess(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr, te = train_df.copy(), test_df.copy()
    for c in NUMS:
        tr[c] = pd.to_numeric(tr[c], errors="coerce").astype(np.float32)
        te[c] = pd.to_numeric(te[c], errors="coerce").astype(np.float32)
        med   = float(np.nanmedian(tr[c].values))
        tr[c] = tr[c].fillna(med)
        te[c] = te[c].fillna(med)
    for c in CATS:
        tr[c] = tr[c].astype(str).str.strip().fillna("missing")
        te[c] = te[c].astype(str).str.strip().fillna("missing")
    y = tr[TARGET].values.astype(np.int64)
    print("Unique labels:", np.unique(y))
    assert not np.isnan(y).any(),          "NaNs in labels!"
    assert set(np.unique(y)) <= {0, 1, 2}, "Invalid labels!"
    return tr, te, y


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def _build_snapper(train_series: pd.Series):
    s  = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = s.value_counts(dropna=False)
    frequent = np.sort(
        np.array([v for v in vc[vc >= RARE_MIN].index if pd.notna(v)], dtype=np.float32)
    )
    if frequent.size == 0:
        frequent = np.sort(s.dropna().unique().astype(np.float32))
    freq_set = set(frequent.tolist())

    def transform(series):
        x       = pd.to_numeric(series, errors="coerce").astype(np.float32).values
        is_nan  = np.isnan(x)
        is_rare = np.ones(len(x), dtype=np.int32)
        for i, v in enumerate(x):
            if not np.isnan(v) and float(v) in freq_set:
                is_rare[i] = 0
        x_snapped = x.copy()
        snap_idx  = np.where((~is_nan) & (is_rare == 1))[0]
        if snap_idx.size > 0 and frequent.size > 0:
            v       = x[snap_idx]
            pos     = np.clip(np.searchsorted(frequent, v), 0, len(frequent) - 1)
            left    = np.clip(pos - 1, 0, len(frequent) - 1)
            nearest = np.where(
                np.abs(v - frequent[pos]) <= np.abs(v - frequent[left]),
                frequent[pos], frequent[left]
            )
            x_snapped[snap_idx] = nearest.astype(np.float32)
        return x_snapped.astype(np.float32), is_rare.astype(np.int32)

    return transform


def engineer_features(train_df, test_df, y_train):
    print("\n[FE] Building features...")
    tr, te = train_df.copy(), test_df.copy()

    for col in NUMS:
        snapper = _build_snapper(tr[col])
        tr_snap, tr_rare = snapper(tr[col])
        te_snap, te_rare = snapper(te[col])
        tr[f"{col}__cat"]     = pd.Series(tr_snap).astype(str).values
        te[f"{col}__cat"]     = pd.Series(te_snap).astype(str).values
        tr[f"{col}__is_rare"] = pd.Series(tr_rare).astype(str).values
        te[f"{col}__is_rare"] = pd.Series(te_rare).astype(str).values

    print("[FE] Fitting decision stumps...")
    stumps = build_decision_stumps(tr, y_train)
    tr     = apply_decision_stumps(tr, stumps)
    te     = apply_decision_stumps(te, stumps)
    for col in NUMS:
        tr[f"{col}__stump"] = tr[f"{col}__stump"].astype(str)
        te[f"{col}__stump"] = te[f"{col}__stump"].astype(str)

    print("[FE] Extracting digit features...")
    tr = add_digit_features(tr, NUMS, k=1)
    te = add_digit_features(te, NUMS, k=1)

    for df in (tr, te):
        for c in ALL_CATS:
            if c in df.columns:
                df[c] = df[c].astype(str).fillna("missing")

    print(f"[FE] Cat cols: {len(ALL_CATS)} | Num: {len(NUMS)} raw + {AE_LATENT_DIM} AE")
    return tr, te, stumps


def encode_categoricals(train_df, test_df):
    print("\n[Encode] Encoding categoricals...")
    tr_codes, te_codes, cardinalities = [], [], []
    for c in ALL_CATS:
        if c not in train_df.columns or c not in test_df.columns:
            continue
        all_vals = pd.concat([train_df[c].astype(str), test_df[c].astype(str)], ignore_index=True)
        mapping  = {v: i for i, v in enumerate(all_vals.unique())}
        tr_codes.append(train_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        te_codes.append(test_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values)
        cardinalities.append(len(mapping))
    Xc_tr = np.stack(tr_codes, axis=1)
    Xc_te = np.stack(te_codes, axis=1)
    print(f"[Encode] train: {Xc_tr.shape} | test: {Xc_te.shape}")
    return Xc_tr, Xc_te, cardinalities


def scale_numerics(train_df, test_df):
    """Rank-Percentile Encoding replaces StandardScaler."""
    print("\n[Scale] Rank-percentile encoding...")
    rpe   = RankPercentileEncoder()
    Xn_tr = rpe.fit_transform(train_df, NUMS)
    Xn_te = rpe.transform(test_df, NUMS)
    print(f"[Scale] train: {Xn_tr.shape} | test: {Xn_te.shape}")
    return Xn_tr, Xn_te, rpe


def build_knn_graph(train_df, test_df, k=K):
    print(f"\n[Graph] Building KNN graph k={k} on {len(train_df)+len(test_df):,} nodes...")
    graph_cat = pd.concat(
        [train_df[GRAPH_CAT_COLS].astype(str), test_df[GRAPH_CAT_COLS].astype(str)],
        ignore_index=True
    )
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
    X_cat_ohe = ohe.fit_transform(graph_cat).astype(np.float32)

    rpe_g = RankPercentileEncoder()
    X_num = np.vstack([
        rpe_g.fit_transform(train_df[GRAPH_NUM_COLS].copy(), GRAPH_NUM_COLS),
        rpe_g.transform(test_df[GRAPH_NUM_COLS].copy(),      GRAPH_NUM_COLS),
    ]).astype(np.float32) * GRAPH_NUM_MULTIPLIER

    X_graph = np.concatenate([X_cat_ohe, X_num], axis=1).astype(np.float32)
    print(f"[Graph] Feature matrix: {X_graph.shape}")
    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(X_graph)
    _, idx = knn.kneighbors(X_graph)
    if hasattr(idx, "get"): idx = idx.get()
    neighbors = idx.astype(np.int32)
    print(f"[Graph] Neighbors: {neighbors.shape}")
    del X_graph, X_cat_ohe, X_num, idx, knn; gc.collect()
    return neighbors


# ============================================================
# MODEL DEFINITION
# ============================================================

def _emb_dim(cardinality: int) -> int:
    return int(np.clip(round(1.8 * (cardinality ** 0.25)), 4, 24))


class CatEmbed(nn.Module):
    def __init__(self, cardinalities):
        super().__init__()
        self.embs    = nn.ModuleList()
        self.out_dim = 0
        for card in cardinalities:
            card = max(2, int(card))
            d    = _emb_dim(card)
            self.embs.append(nn.Embedding(card, d))
            self.out_dim += d
        for e in self.embs:
            nn.init.normal_(e.weight, 0.0, 0.02)

    def forward(self, x_cat):
        return torch.cat([emb(x_cat[:, j]) for j, emb in enumerate(self.embs)], dim=1)


class IrrigationGNN(nn.Module):
    """2-layer GraphSAGE + categorical embeddings + residual connections."""
    def __init__(self, num_in, cardinalities, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.cat    = CatEmbed(cardinalities)
        in_dim      = num_in + self.cat.out_dim
        self.lin_in = nn.Linear(in_dim, hidden)
        self.conv1  = SAGEConv(hidden, hidden)
        self.conv2  = SAGEConv(hidden, hidden)
        self.norm1  = nn.LayerNorm(hidden)
        self.norm2  = nn.LayerNorm(hidden)
        self.drop   = dropout
        self.head   = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(inplace=True),
            nn.Dropout(dropout),   nn.Linear(64, N_CLASSES),
        )

    def forward(self, data):
        x  = torch.cat([data.x_num, self.cat(data.x_cat)], dim=1)
        x  = F.dropout(F.relu(self.lin_in(x)), p=self.drop, training=self.training)
        x1 = F.relu(self.norm1(self.conv1(x,  data.edge_index)))
        x1 = F.dropout(x1, p=self.drop, training=self.training)
        x  = x + 0.5 * x1
        x2 = F.relu(self.norm2(self.conv2(x, data.edge_index)))
        x2 = F.dropout(x2, p=self.drop, training=self.training)
        x  = x + 0.5 * x2
        return self.head(x)


# ============================================================
# SUBGRAPH UTILITIES
# ============================================================

_global_pos = None


def _build_subgraph(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu,
                    fanouts, device, offset=0):
    global _global_pos
    seed_nodes = np.asarray(seed_nodes, dtype=np.int32)
    frontier   = seed_nodes
    collected  = [seed_nodes]
    for hop, fanout in enumerate(fanouts):
        nbr      = neighbors[frontier]
        start    = (offset + hop) % nbr.shape[1]
        cols     = (np.arange(fanout) + start) % nbr.shape[1]
        frontier = np.unique(nbr[:, cols].reshape(-1))
        collected.append(frontier)
    nodes = np.unique(np.concatenate(collected))
    m     = len(nodes)
    _global_pos[nodes] = np.arange(m, dtype=np.int32)
    sub_nbr    = neighbors[nodes]
    dst_local  = _global_pos[sub_nbr]
    mask       = dst_local >= 0
    src_l      = np.repeat(np.arange(m, dtype=np.int64), sub_nbr.shape[1])[mask.reshape(-1)]
    dst_l      = dst_local[mask].astype(np.int64)
    edge_index = torch.tensor(np.vstack([src_l, dst_l]), dtype=torch.long, device=device)
    batch = Data(
        x_num      = x_num_cpu[nodes].to(device),
        x_cat      = x_cat_cpu[nodes].to(device, non_blocking=True),
        y          = y_cpu[nodes].to(device, non_blocking=True),
        edge_index = edge_index,
    )
    batch.seed_local = torch.tensor(_global_pos[seed_nodes], dtype=torch.long, device=device)
    _global_pos[nodes] = -1
    return batch


def _seed_batches(seed_nodes, batch_size, shuffle):
    arr = np.asarray(seed_nodes, dtype=np.int32).copy()
    if shuffle: np.random.shuffle(arr)
    for i in range(0, len(arr), batch_size):
        yield arr[i:i + batch_size]


@torch.no_grad()
def _predict_nodes(model, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu,
                   fanouts, batch_size, device, offset=0):
    model.eval()
    out = np.zeros((len(seed_nodes), N_CLASSES), dtype=np.float32)
    pos = 0
    for batch_seeds in _seed_batches(seed_nodes, batch_size, shuffle=False):
        batch  = _build_subgraph(batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                                  y_cpu, fanouts, device, offset)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(USE_AMP and device.type == "cuda")):
            logits = model(batch)
        probs = F.softmax(logits[batch.seed_local], dim=1).float().cpu().numpy()
        out[pos:pos + len(batch_seeds)] = probs
        pos += len(batch_seeds)
        del batch, logits, probs
    return out


# ============================================================
# SKLEARN WRAPPER
# ============================================================

class IrrigationGNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        hidden        = HIDDEN,
        dropout       = DROPOUT,
        lr            = LR,
        weight_decay  = WEIGHT_DECAY,
        epochs        = EPOCHS,
        patience      = PATIENCE,
        batch_size    = BATCH_SIZE,
        infer_batch   = INFER_BATCH,
        fanouts       = FANOUTS,
        device        = DEVICE,
        use_amp       = USE_AMP,
        n_gpus        = None,
        early_stop_on = "val_ba",   # 'val_ba' or 'val_loss'
    ):
        self.hidden        = hidden
        self.dropout       = dropout
        self.lr            = lr
        self.weight_decay  = weight_decay
        self.epochs        = epochs
        self.patience      = patience
        self.batch_size    = batch_size
        self.infer_batch   = infer_batch
        self.fanouts       = fanouts
        self.device        = device
        self.use_amp       = use_amp
        self.early_stop_on = early_stop_on
        self.model_        = None
        self.cardinalities_= None
        self.n_gpus        = n_gpus if n_gpus is not None else torch.cuda.device_count()

    def fit(self, train_idx, val_idx, y_all, neighbors,
            x_num_cpu, x_cat_cpu, y_cpu, cardinalities, class_weights):

        global _global_pos
        n_all       = x_num_cpu.shape[0]
        _global_pos = np.full(n_all, -1, dtype=np.int32)
        self.cardinalities_ = cardinalities

        core_model = IrrigationGNN(
            num_in        = x_num_cpu.shape[1],
            cardinalities = cardinalities,
            hidden        = self.hidden,
            dropout       = self.dropout,
        ).to(self.device)

        # F5 — SSL pre-training: warm-start encoder before supervised CV
        if USE_SSL_PRETRAIN:
            core_model  = pretrain_ssl(
                model      = core_model,
                x_num_cpu  = x_num_cpu,
                x_cat_cpu  = x_cat_cpu,
                neighbors  = neighbors,
                n_all      = n_all,
                device     = self.device,
            )
            _global_pos = np.full(n_all, -1, dtype=np.int32)  # reset after SSL

        if self.n_gpus > 1:
            core_model = nn.DataParallel(core_model)
        self.model_ = core_model.to(self.device)

        # F2 — Label Smoothing: non-zero smoothing softens overconfident Low predictions
        loss_fn = nn.CrossEntropyLoss(
            weight          = class_weights.to(self.device),
            label_smoothing = LABEL_SMOOTHING if USE_LABEL_SMOOTH else 0.0,
        )

        opt = torch.optim.AdamW(
            self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )

        # F4 — Cosine Annealing: decays LR from LR to COSINE_ETA_MIN over EPOCHS
        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=self.epochs, eta_min=COSINE_ETA_MIN
            ) if USE_COSINE_LR else None
        )

        scaler     = torch.cuda.amp.GradScaler(
            enabled=(self.use_amp and self.device.type == "cuda")
        )
        ba_tracker = BalancedAccuracyTracker(n_classes=N_CLASSES)

        best_val_metric = -float("inf") if self.early_stop_on == "val_ba" else float("inf")
        best_state      = None
        bad_epochs      = 0

        flags_on = (
            ([f"LabelSmooth={LABEL_SMOOTHING}"] if USE_LABEL_SMOOTH else []) +
            ([f"Mixup(a={MIXUP_ALPHA})"]         if USE_MIXUP        else []) +
            (["CosineAnnealing"]                 if USE_COSINE_LR    else []) +
            (["SSL_pretrained"]                  if USE_SSL_PRETRAIN else []) +
            (["TTA"]                             if USE_TTA          else [])
        )
        print(f"    Params : {sum(p.numel() for p in self.model_.parameters()):,}")
        print(f"    Train  : {len(train_idx):,}  Val: {len(val_idx):,}")
        print(f"    Flags  : {', '.join(flags_on) if flags_on else 'none (base config)'}")
        print(f"    Stop on: {self.early_stop_on}  Patience: {self.patience}\n")

        for epoch in range(1, self.epochs + 1):
            self.model_.train()
            epoch_losses = []
            offset       = epoch % K

            for batch_seeds in _seed_batches(train_idx, self.batch_size, shuffle=True):
                batch = _build_subgraph(
                    batch_seeds, neighbors, x_num_cpu, x_cat_cpu,
                    y_cpu, self.fanouts, self.device, offset
                )
                opt.zero_grad(set_to_none=True)

                with torch.autocast(device_type="cuda", dtype=torch.float16,
                                    enabled=(self.use_amp and self.device.type == "cuda")):

                    if USE_MIXUP:
                        # F3 — Mixup: blend seed-node features in embedding space.
                        # We operate post-cat-embedding but pre-lin_in so the graph
                        # edges remain intact.  Only seed nodes are mixed; context
                        # nodes keep their original features for message passing.
                        _m = (self.model_.module
                              if isinstance(self.model_, nn.DataParallel)
                              else self.model_)

                        seed_loc    = batch.seed_local
                        x_num_seed  = batch.x_num[seed_loc]
                        cat_emb_all = _m.cat(batch.x_cat)
                        x_cat_seed  = cat_emb_all[seed_loc]
                        y_seed      = batch.y[seed_loc].long()

                        x_num_mix, x_cat_mix, lam, y_a, y_b = _mixup_batch(
                            x_num_seed, x_cat_seed, y_seed
                        )

                        # Splice mixed seed rows back into full subgraph tensors
                        x_num_full           = batch.x_num.clone()
                        x_num_full[seed_loc] = x_num_mix
                        cat_emb_full           = cat_emb_all.clone()
                        cat_emb_full[seed_loc] = x_cat_mix

                        # Forward using encoder manually (cat already computed)
                        x  = torch.cat([x_num_full, cat_emb_full], dim=1)
                        x  = F.dropout(F.relu(_m.lin_in(x)), p=self.dropout, training=True)
                        x1 = F.relu(_m.norm1(_m.conv1(x,  batch.edge_index)))
                        x1 = F.dropout(x1, p=self.dropout, training=True)
                        x  = x + 0.5 * x1
                        x2 = F.relu(_m.norm2(_m.conv2(x, batch.edge_index)))
                        x2 = F.dropout(x2, p=self.dropout, training=True)
                        x  = x + 0.5 * x2
                        logits = _m.head(x)
                        loss   = _mixup_loss(loss_fn, logits[seed_loc], lam, y_a, y_b)

                    else:
                        logits = self.model_(batch)
                        loss   = loss_fn(
                            logits[batch.seed_local],
                            batch.y[batch.seed_local].long()
                        )

                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(self.model_.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                epoch_losses.append(loss.item())
                del batch, logits, loss

            # F4 — step scheduler after each full epoch
            if scheduler is not None:
                scheduler.step()

            # ---- validate ----
            val_probs  = _predict_nodes(
                self.model_, val_idx, neighbors, x_num_cpu, x_cat_cpu,
                y_cpu, self.fanouts, self.infer_batch, self.device, offset
            )
            y_val_true = y_all[val_idx]
            val_loss   = log_loss(y_val_true, val_probs, labels=[0, 1, 2])
            ba_tracker.reset()
            ba_tracker.update(val_probs, y_val_true)
            val_ba = ba_tracker.compute()

            lr_str = (f" | LR: {scheduler.get_last_lr()[0]:.2e}"
                      if USE_COSINE_LR else "")
            print(
                f"    Epoch {epoch:04d} | "
                f"Train Loss: {np.mean(epoch_losses):.5f} | "
                f"Val LogLoss: {val_loss:.5f} | "
                f"Val BA: {val_ba:.5f}{lr_str}"
            )

            # ---- early stopping ----
            if self.early_stop_on == "val_ba":
                improved = val_ba   > best_val_metric + 1e-6
                current  = val_ba
            else:
                improved = val_loss < best_val_metric - 1e-6
                current  = val_loss

            if improved:
                best_val_metric = current
                best_state      = {k: v.detach().cpu().clone()
                                   for k, v in self.model_.state_dict().items()}
                bad_epochs      = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    print(f"\n    [Early Stop] epoch {epoch}  patience={self.patience}")
                    break

        label = "Best Val BA" if self.early_stop_on == "val_ba" else "Best Val LogLoss"
        print(f"\n    [{label}]: {best_val_metric:.5f}")
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    # ----------------------------------------------------------
    def predict_proba(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        """Standard single-pass inference."""
        return _predict_nodes(
            self.model_, seed_nodes, neighbors, x_num_cpu, x_cat_cpu,
            y_cpu, self.fanouts, self.infer_batch, self.device
        )

    def predict_proba_tta(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu,
                          y_cpu, n_passes=TTA_PASSES):
        """
        F1 — Test-Time Augmentation: average n_passes runs with different
        neighborhood offsets.

        Currently USE_TTA=False because high FANOUT/K (6/8) leaves little
        sampling variance between passes.  To re-enable meaningfully:
          1. Set USE_TTA = True
          2. Reduce fanouts, e.g. FANOUTS = [4, 3], to increase per-pass diversity
        """
        acc = np.zeros((len(seed_nodes), N_CLASSES), dtype=np.float64)
        for i in range(n_passes):
            offset = (i * K // n_passes) % K
            acc   += _predict_nodes(
                self.model_, seed_nodes, neighbors, x_num_cpu, x_cat_cpu,
                y_cpu, self.fanouts, self.infer_batch, self.device, offset=offset
            )
        return (acc / n_passes).astype(np.float32)

    def _infer(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        """Routes to TTA or standard inference based on the USE_TTA flag."""
        if USE_TTA:
            return self.predict_proba_tta(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu)
        return self.predict_proba(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu)

    def predict(self, seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu):
        return np.argmax(
            self.predict_proba(seed_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu), axis=1
        )


# ============================================================
# SECTION 7 — LOAD DATA & BUILD GRAPH
# ============================================================

print("\n" + "="*60)
print("Loading data...")
print("="*60)

TRAIN_PARQUET = "/content/drive/MyDrive/irrigation_need_v15/train_engineered_v21.parquet"
TEST_PARQUET  = "/content/drive/MyDrive/irrigation_need_v15/test_engineered_v21.parquet"

train_raw = pd.read_parquet(TRAIN_PARQUET)
test_raw  = pd.read_parquet(TEST_PARQUET)
print(f"Raw train: {train_raw.shape} | Raw test: {test_raw.shape}")

train_pre, test_pre, y_train = preprocess(train_raw, test_raw)
train_fe,  test_fe,  stumps  = engineer_features(train_pre, test_pre, y_train)

Xc_train, Xc_test, cat_cardinalities = encode_categoricals(train_fe, test_fe)
Xn_train_raw, Xn_test_raw, rpe       = scale_numerics(train_fe, test_fe)
ae_train, ae_test                    = train_autoencoder(Xn_train_raw, Xn_test_raw)

Xn_train = np.concatenate([Xn_train_raw, ae_train], axis=1).astype(np.float32)
Xn_test  = np.concatenate([Xn_test_raw,  ae_test],  axis=1).astype(np.float32)
print(f"[Features] Numeric dim: {Xn_train.shape[1]} (raw={len(NUMS)} + AE={AE_LATENT_DIM})")

neighbors = build_knn_graph(train_fe, test_fe, k=K)

n_train = len(train_fe)
n_test  = len(test_fe)
n_all   = n_train + n_test

Xn_all   = np.vstack([Xn_train, Xn_test])
Xc_all   = np.vstack([Xc_train, Xc_test])
y_all_np = np.concatenate([y_train, np.full(n_test, -1, dtype=np.int64)])

x_num_cpu = torch.tensor(Xn_all,   dtype=torch.float32).pin_memory()
x_cat_cpu = torch.tensor(Xc_all,   dtype=torch.long).pin_memory()
y_cpu     = torch.tensor(y_all_np, dtype=torch.long).pin_memory()

print(f"\n[Tensors] x_num: {tuple(x_num_cpu.shape)} | x_cat: {tuple(x_cat_cpu.shape)}")

classes       = np.arange(N_CLASSES)
cw_values     = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = torch.tensor(cw_values, dtype=torch.float32)
print(f"\n[Weights] { {LABEL_INV[i]: round(float(w), 4) for i, w in enumerate(cw_values)} }")


# ============================================================
# SECTION 8 — CROSS-VALIDATION
# ============================================================

print("\n" + "="*60)
print("Starting 5-Fold Cross-Validation")
print("="*60)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs  = np.zeros((n_train, N_CLASSES), dtype=np.float32)
pred_probs = np.zeros((n_test,  N_CLASSES), dtype=np.float32)
test_nodes = np.arange(n_train, n_all, dtype=np.int32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), y_train), 1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold} / {N_FOLDS}")
    print(f"{'='*60}\n")
    t_fold = time.time()

    clf = IrrigationGNNClassifier(early_stop_on="val_ba")
    clf.fit(
        train_idx    = tr_idx,
        val_idx      = va_idx,
        y_all        = y_all_np,
        neighbors    = neighbors,
        x_num_cpu    = x_num_cpu,
        x_cat_cpu    = x_cat_cpu,
        y_cpu        = y_cpu,
        cardinalities= cat_cardinalities,
        class_weights= class_weights,
    )

    # _infer() automatically routes to TTA or standard based on USE_TTA
    oof_probs[va_idx] = clf._infer(va_idx,     neighbors, x_num_cpu, x_cat_cpu, y_cpu)
    pred_probs       += clf._infer(test_nodes, neighbors, x_num_cpu, x_cat_cpu, y_cpu) / N_FOLDS

    fold_ba = balanced_accuracy_score(y_train[va_idx], np.argmax(oof_probs[va_idx], axis=1))
    fold_ll = log_loss(y_train[va_idx], oof_probs[va_idx], labels=[0, 1, 2])
    print(f"\n  [Fold {fold}] BA: {fold_ba:.5f} | LogLoss: {fold_ll:.5f} | "
          f"Time: {time.time()-t_fold:.1f}s")

    del clf; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


# ============================================================
# SECTION 9 — OOF EVALUATION & SAVE
# ============================================================

oof_preds  = np.argmax(oof_probs, axis=1)
cv_bal_acc = balanced_accuracy_score(y_train, oof_preds)
cv_logloss = log_loss(y_train, oof_probs, labels=[0, 1, 2])

print("\n" + "="*60)
print("  OOF EVALUATION")
print("="*60)
print(f"  CV Balanced Accuracy : {cv_bal_acc:.5f}")
print(f"  CV Log-Loss          : {cv_logloss:.5f}")

# Threshold optimisation (directly maximises BA)
best_thresholds  = optimize_thresholds(oof_probs, y_train)
oof_preds_thresh = apply_thresholds(oof_probs, best_thresholds)
ba_thresh        = balanced_accuracy_score(y_train, oof_preds_thresh)
print(f"  Threshold BA         : {ba_thresh:.5f}  (gain: {ba_thresh - cv_bal_acc:+.5f})")

test_preds_thresh = apply_thresholds(pred_probs, best_thresholds)

# ---- Standard saves (always) ----
np.save(f"{OUT_DIR}/thresholds_{VERSION_NB}.npy",        best_thresholds)
np.save(f"{OUT_DIR}/test_preds_thresh_{VERSION_NB}.npy", test_preds_thresh)
np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy",               oof_probs)
np.save(f"{OUT_DIR}/test_preds_{VERSION_NB}.npy",        pred_probs)
np.save(f"{OUT_DIR}/y_train_{VERSION_NB}.npy",           y_train)

# F6 — Stacker OOF: raw softmax probabilities for the LGB meta-learner.
# These must be pre-threshold, pre-tuning — the stacker sees calibrated
# raw outputs.  Threshold tuning is a submission-time operation only.
if SAVE_STACKER_OOF:
    oof_path  = f"{OUT_DIR}/oof_stacker_{VERSION_NB}.npy"
    pred_path = f"{OUT_DIR}/pred_stacker_{VERSION_NB}.npy"
    np.save(oof_path,  oof_probs)
    np.save(pred_path, pred_probs)
    print(f"\n[F6] Stacker OOF  -> {oof_path}   shape={oof_probs.shape}  dtype={oof_probs.dtype}")
    print(f"[F6] Stacker pred -> {pred_path}  shape={pred_probs.shape}  dtype={pred_probs.dtype}")
    print(f"     Column order: [Low=0, Medium=1, High=2]  (matches NEEDS_REMAP conventions)")


# ============================================================
# SECTION 10 — SUBMISSION
# ============================================================

print("\n[Submission] Generating predictions...")

test_preds  = np.argmax(pred_probs, axis=1)
test_labels = [LABEL_INV[p] for p in test_preds]

submission  = pd.DataFrame({"id": test_raw["id"], TARGET: test_labels})
submission.to_csv(f"submission_gnn_{VERSION_NB}.csv", index=False)
print(f"Saved: submission_gnn_{VERSION_NB}.csv")
print(train_raw[TARGET].value_counts())
print("Unique labels in y_train:", np.unique(y_train))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in links: https://data.pyg.org/whl/torch-2.4.0+cu121.html
Using RAPIDS cuML KNN - Very Fast!
DEVICE: cuda

  ACTIVE FEATURE FLAGS
  OFF  USE_TTA
  ON   USE_LABEL_SMOOTH
  ON   USE_MIXUP
  ON   USE_COSINE_LR
  ON   USE_SSL_PRETRAIN
  ON   SAVE_STACKER_OOF


Loading data...
Raw train: (640000, 111) | Raw test: (270000, 110)
Unique labels: [0 1 2]

[FE] Building features...
[FE] Fitting decision stumps...
[FE] Extracting digit features...
[FE] Cat cols: 52 | Num: 11 raw + 4 AE

[Encode] Encoding categoricals...
[Encode] train: (640000, 52) | test: (270000, 52)

[Scale] Rank-percentile encoding...
[Scale] train: (640000, 11) | test: (270000, 11)

[AE] Training  latent=4  epochs=30
  AE Epoch 001 | MSE: 0.075290
  AE Epoch 005 | MSE: 0.045217
  AE Epoch 010 | MSE: 0.039351
  AE Epoch 015 | MSE: 0.037911
  AE Epoch 020 | MSE: 0.037214
  AE Epoch 025 | MSE: 

In [10]:
np.save(f"{OUT_DIR}/oof_{VERSION_NB}.npy", oof_probs)
np.save(f"{OUT_DIR}/test_preds_{VERSION_NB}.npy", pred_probs)
np.save(f"{OUT_DIR}/y_train_{VERSION_NB}.npy", y_train)

In [9]:
print(train_raw[TARGET].value_counts())

Irrigation_Need
0    375781
1    242874
2     21345
Name: count, dtype: int64


In [8]:
print("Unique labels in y_train:", np.unique(y_train))

Unique labels in y_train: [-9223372036854775808]
